# Yatharth Music AI — Kaggle Free GPU launcher

**Recommended $0 test/validation path.** This notebook runs ACE-Step 1.5 + Yatharth Music AI in a Kaggle GPU notebook and exposes Yatharth temporarily through Cloudflare Tunnel.

Kaggle GPU availability is subject to account/platform limits. This is for free validation and early testing, not guaranteed permanent hosting.

**Before running:** Kaggle Notebook → Settings → Accelerator → GPU. If Kaggle asks for Internet access, enable Internet. Run cells top-to-bottom.

In [ ]:
!nvidia-smi || true
!rm -rf /kaggle/working/ACE-Step-1.5 /kaggle/working/yatharth-music-ai
!git clone --depth 1 https://github.com/ace-step/ACE-Step-1.5.git /kaggle/working/ACE-Step-1.5
!git clone --depth 1 https://github.com/rampaulsaini/yatharth-music-ai.git /kaggle/working/yatharth-music-ai
%cd /kaggle/working/ACE-Step-1.5
!pip -q install uv
!uv sync --frozen
%cd /kaggle/working/yatharth-music-ai
!pip -q install -r requirements.txt

In [ ]:
import subprocess, time, os, requests
log = open('/kaggle/working/acestep.log', 'w')
proc = subprocess.Popen(['uv','run','python','-m','acestep.api_server','--host','127.0.0.1','--port','8001'], stdout=log, stderr=subprocess.STDOUT, cwd='/kaggle/working/ACE-Step-1.5')
ready = False
last_error = None
for _ in range(120):
    time.sleep(2)
    if proc.poll() is not None: break
    try:
        r = requests.get('http://127.0.0.1:8001/health', timeout=5)
        print('ACE-Step:', r.status_code, r.text[:500])
        if r.status_code < 500:
            ready = True; break
    except Exception as e: last_error = str(e)
print('ACE-Step READY:', ready, 'PID:', proc.pid, 'exit:', proc.poll())
if not ready:
    print('Last connection error:', last_error)
    print(open('/kaggle/working/acestep.log', errors='ignore').read()[-12000:])
    raise RuntimeError('ACE-Step did not become ready.')

In [ ]:
import subprocess, time, requests, os
ylog = open('/kaggle/working/yatharth.log', 'w')
env = os.environ.copy(); env['DEMO_MODE']='false'; env['MUSIC_ENGINE_URL']='http://127.0.0.1:8001'; env['TRUST_PROXY']='false'
backend = subprocess.Popen(['python','-m','uvicorn','main:app','--host','0.0.0.0','--port','8000'], stdout=ylog, stderr=subprocess.STDOUT, cwd='/kaggle/working/yatharth-music-ai', env=env)
ready = False
for _ in range(30):
    time.sleep(2)
    try:
        r=requests.get('http://127.0.0.1:8000/api/health',timeout=10); body=r.json(); print(body)
        if r.status_code==200 and body.get('ok') and body.get('engine_reachable'):
            ready=True; break
    except Exception as e: print('Waiting for Yatharth:',e)
print('Yatharth READY:', ready, 'PID:', backend.pid)
if not ready:
    print(open('/kaggle/working/yatharth.log', errors='ignore').read()[-12000:])
    raise RuntimeError('Yatharth is not ready or cannot reach ACE-Step.')

In [ ]:
import subprocess, re, time
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
tunnel_log=open('/kaggle/working/cloudflared.log','w')
tunnel=subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8000'],stdout=tunnel_log,stderr=subprocess.STDOUT)
public_url=None
for _ in range(45):
    time.sleep(2); text=open('/kaggle/working/cloudflared.log',errors='ignore').read(); match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',text)
    if match: public_url=match.group(0); break
print('YATHARTH PUBLIC LINK:',public_url)
if not public_url:
    print(open('/kaggle/working/cloudflared.log',errors='ignore').read()[-8000:]); raise RuntimeError('Cloudflare public URL was not created.')
print('Keep this Kaggle notebook session running while testing from the phone.')

## Real AI test

Open the printed **YATHARTH PUBLIC LINK** on your phone. Start with a **10–30 second** generation. Then try 60 seconds if stable.

A successful test must produce actual ACE-Step audio; the demo test tone does not count.

In [ ]:
print('--- ACE-Step log ---'); print(open('/kaggle/working/acestep.log',errors='ignore').read()[-12000:])
print('--- Yatharth log ---'); print(open('/kaggle/working/yatharth.log',errors='ignore').read()[-12000:])
print('--- Cloudflare log ---'); print(open('/kaggle/working/cloudflared.log',errors='ignore').read()[-8000:])